# Job Scraping Analysis Notebook
**Name**: Mayenmein Terence Sama Aloah Jr<br>
**Date**: October 2025<br>
**Project**: SkillHub Job Data Collection
## Introduction
This notebook demonstrates the functionality of the JobScraper class for collecting job posting data from the Found.dev API. The implementation focuses on batch processing, memory efficiency, and progress tracking.

## 1. Import and Setup
Let's start by importing the necessary modules and setting up our environment.

In [1]:
import sys
import os
import pandas as pd
from datetime import datetime
from pathlib import Path

# Add the src directory to the path to import our custom module
sys.path.append('..')

# Import the JobScraper class
from src.scrape_jobs import JobScraper

print("✅ Imports completed successfully!")
print(f"📅 Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports completed successfully!
📅 Analysis date: 2025-10-22 12:42:29


## 2. Initialize the Job Scraper
Create an instance of the JobScraper class with custom configuration.

In [2]:
# Initialize the scraper with custom output path
OUTPUT_PATH = Path('../data/raw/combined_jobs.csv')
scraper = JobScraper(output_file=OUTPUT_PATH)

print(f"🎯 Scraper initialized successfully!")
print(f"📁 Output file: {scraper.output_file}")
print(f"🌐 API endpoint: {scraper.BASE_URL}")

🎯 Scraper initialized successfully!
📁 Output file: ..\data\raw\combined_jobs.csv
🌐 API endpoint: https://api.found.dev/api/open/jobs


## 3. Test Single Page Fetch
Before running full batch scraping, let's test fetching a single page to understand the data structure.

In [3]:
def test_single_fetch():
    """Test fetching a single page of job data"""
    print("🔍 Testing single page fetch...")
    
    try:
        # Fetch first page
        data = scraper.fetch_jobs(page=1, skill="Data Science", ai=True)
        jobs = data.get("jobs", [])
        
        print(f"📄 Jobs found on page 1: {len(jobs)}")
        
        if jobs:
            # Process the jobs
            processed_jobs = scraper.process_job_data(jobs[:2])  # Process first 2 jobs as sample
            print(f"🔄 Processed jobs sample: {len(processed_jobs)}")
            
            # Display sample data
            if processed_jobs:
                sample_df = pd.DataFrame(processed_jobs)
                print("\n📊 Sample job data structure:")
                print(sample_df[['title', 'company', 'location']].head())
                
        return len(jobs)
        
    except Exception as e:
        print(f"❌ Error during test fetch: {e}")
        return 0

# Run the test
jobs_count = test_single_fetch()
print(f"\n✅ Single page test completed. Found {jobs_count} jobs.")

🔍 Testing single page fetch...
📄 Jobs found on page 1: 100
🔄 Processed jobs sample: 2

📊 Sample job data structure:
                                               title            company  \
0   Senior Python Data Engineer (Finance Technology)         Crypto.com   
1  Flight Data Analytics Engineer | Digital Infra...  BETA Technologies   

              location  
0      Shenzhen, China  
1  Burlington, VT, USA  

✅ Single page test completed. Found 100 jobs.


## 4. Run Small Batch Scraping
Now let's run a small batch scraping operation to demonstrate the functionality.

In [4]:
def run_small_batch_scraping():
    """Run scraping with a small batch size for demonstration"""
    print("🚀 Starting small batch scraping...")
    
    # Run with small batch size for quick demonstration
    total_jobs = scraper.scrape_in_batches(
        skill="Data Science",
        pages_per_batch=3,
        ai=True,
        delay=1,
        max_batches=2
    )
    
    print(f"\n🎉 Small batch scraping completed!")
    print(f"📊 Total jobs collected: {total_jobs}")
    
    return total_jobs

# Execute small batch scraping
small_batch_total = run_small_batch_scraping()

🚀 Starting small batch scraping...
Scraping Data Science jobs...


Batches: 1batch [01:05, 65.06s/batch, total=300]

Reached maximum batch limit: 2

🎉 Small batch scraping completed!
📊 Total jobs collected: 300


## 5. Analyze Collected Data
Let's analyze the data we've collected so far.

In [6]:
def analyze_collected_data():
    """Analyze the collected job data"""
    print("📈 Analyzing collected data...")
    
    if os.path.exists(scraper.output_file):
        # Load the data
        df = pd.read_csv(scraper.output_file)
        
        print(f"📁 Data file: {scraper.output_file}")
        print(f"📊 Total records: {len(df)}")
        print(f"🏢 Unique companies: {df['company'].nunique()}")
        print(f"📍 Job locations distribution:")
        print(df['location'].value_counts().head(10))
        
        # Display recent jobs
        print("\n🆕 Most recent jobs:")
        if 'published' in df.columns:
            recent_jobs = df.sort_values('published', ascending=False).head(5)
            print(recent_jobs[['title', 'company', 'published']])
        else:
            print(df[['title', 'company']].head(5))
            
        return df
    else:
        print("❌ No data file found. Please run the scraper first.")
        return None

# Analyze the data
job_data = analyze_collected_data()

📈 Analyzing collected data...
📁 Data file: ..\data\raw\combined_jobs.csv
📊 Total records: 300
🏢 Unique companies: 156
📍 Job locations distribution:
location
, USA                     26
Bangalore, India          17
New York, NY, USA         15
San Francisco, CA, USA    11
Hyderabad, India           9
London, UK                 9
Mountain View, CA, USA     9
Pune, India                7
Redmond, WA, USA           7
, India                    6
Name: count, dtype: int64

🆕 Most recent jobs:
                                               title              company  \
0  Player Protection and Game Security Data Scien...              Ubisoft   
1                                        AI Engineer             Syngenta   
2                                     AI/ML Engineer  Renesas Electronics   
3                         Data Science and AI Expert         Sopra Steria   
4                                        AI Engineer         Sopra Steria   

                     published  
0  2025-10

## 6. Full-Scale Scraping
For comprehensive data collection, run the full scraping process.

In [ ]:
def run_comprehensive_scraping():
    """Run comprehensive job scraping"""
    print("🔍 Starting comprehensive scraping...")
    
    # You can adjust these parameters based on your needs
    total_jobs = scraper.scrape_in_batches(
        skill="Data Science",
        pages_per_batch=20,  # Larger batches for efficiency
        ai=True,
        delay=1,
        max_batches=None  # No limit, stops when no more jobs
    )
    
    print(f"\n🏁 Comprehensive scraping completed!")
    print(f"📈 Total jobs collected: {total_jobs}")
    
    return total_jobs

# Uncomment to run comprehensive scraping (may take time)
# comprehensive_total = run_comprehensive_scraping()

## 7. Data Quality Check
Perform data quality checks on the collected dataset.

In [7]:
def data_quality_report():
    """Generate a data quality report"""
    print("🔍 Generating data quality report...")
    
    if not os.path.exists(scraper.output_file):
        print("❌ No data file found.")
        return
    
    df = pd.read_csv(scraper.output_file)
    
    print("📊 DATA QUALITY REPORT")
    print("=" * 50)
    
    # Basic statistics
    print(f"Total records: {len(df):,}")
    print(f"Data columns: {list(df.columns)}")
    
    # Missing values analysis
    print("\n❓ MISSING VALUES ANALYSIS:")
    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    
    for col in df.columns:
        print(f"  {col}: {missing_data[col]:,} missing ({missing_percent[col]:.1f}%)")
    
    # Data completeness
    complete_records = df.notnull().all(axis=1).sum()
    print(f"\n✅ Complete records: {complete_records:,} ({complete_records/len(df)*100:.1f}%)")
    
    return df

# Generate quality report
quality_df = data_quality_report()

🔍 Generating data quality report...
📊 DATA QUALITY REPORT
Total records: 300
Data columns: ['title', 'company', 'city', 'country', 'location', 'skills', 'type', 'salary', 'salary_min', 'salary_max', 'published', 'ai']

❓ MISSING VALUES ANALYSIS:
  title: 0 missing (0.0%)
  company: 0 missing (0.0%)
  city: 50 missing (16.7%)
  country: 4 missing (1.3%)
  location: 0 missing (0.0%)
  skills: 0 missing (0.0%)
  type: 1 missing (0.3%)
  salary: 160 missing (53.3%)
  salary_min: 0 missing (0.0%)
  salary_max: 0 missing (0.0%)
  published: 0 missing (0.0%)
  ai: 0 missing (0.0%)

✅ Complete records: 117 (39.0%)


## 8. Summary and Next Steps
### Key Features Demonstrated:
- ✅ Batch processing with memory management
- ✅ Progress tracking with tqdm
- ✅ Data validation and cleaning
- ✅ Error handling and recovery
- ✅ Flexible configuration

### Usage Notes:
- The scraper automatically handles pagination
- Memory is cleared after each batch
- Progress is displayed with visual bars
- Data is saved incrementally